#### The code below is a function that compares two Random Forest models with different feature sets against three baselines.
Run it first before doing the other things

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import itertools

# ==========================================
# 1. DATA LOADING & BASELINE PREP
# ==========================================
match   = pd.read_csv("../data/gold_match_with_league_standings.csv")
tickets = pd.read_csv("../data/gold_match_tickets.csv")
context = pd.read_csv("../data/gold_match_context.csv")

match = match[match["is_home_match"] == True]
df = match.merge(tickets, on="match_id", how="left")
df = df.merge(context, on="match_id", how="left")

# Handle potential suffix issues
if "match_date_x" in df.columns:
    df = df.rename(columns={"match_date_x": "match_date"})
elif "match_date_y" in df.columns:
    df = df.rename(columns={"match_date_y": "match_date"})

df = df[df["away_team"] != "OH Leuven"]
df = df.dropna(subset=["tickets_scanned", "last_result_vs_opponent"])

df["match_date"] = pd.to_datetime(df["match_date"])
df = df.sort_values("match_date").reset_index(drop=True)

# --- REQUIRED BASELINE CALCULATIONS (The function needs these) ---

# 1. Opponent Average Baseline
df["opponent_avg"] = df.groupby("away_team")["tickets_scanned"].transform(lambda x: x.shift().expanding().mean())
df["season_expanding_avg"] = df["tickets_scanned"].shift().expanding().mean()
df["opponent_avg"] = df["opponent_avg"].fillna(df["season_expanding_avg"])

# 2. Same Fixture Baseline
def get_last_fixture(row, dataframe):
    past = dataframe[(dataframe["away_team"] == row["away_team"]) & (dataframe["match_date"] < row["match_date"])]
    if not past.empty:
        return past.iloc[-1]["tickets_scanned"]
    return np.nan

df["same_fixture_baseline"] = df.apply(lambda r: get_last_fixture(r, df), axis=1)
df["same_fixture_baseline"] = df["same_fixture_baseline"].fillna(df["season_expanding_avg"])

# Cleanup Helper
def parse_result(value):
    if pd.isna(value): return None
    if value.startswith("W"): return 3
    elif value.startswith("D"): return 1
    elif value.startswith("L"): return 0
    return None
df["last_result_numeric"] = df["last_result_vs_opponent"].apply(parse_result)

# ==========================================
# 1. CORE EVALUATION FUNCTION
# ==========================================
def compare_feature_sets(df, features_1, features_2, name_1="Model 1", name_2="Model 2", verbose=True):
    """Evaluates two feature sets via walk-forward validation and returns a results DataFrame."""
    if "match_date" in df.columns:
        df = df.sort_values("match_date").reset_index(drop=True)
        
    seasons = sorted(df["season"].unique())
    season_results = []
    
    required_cols = list(set(features_1 + features_2 + ["tickets_scanned", "season", "opponent_avg", "same_fixture_baseline"]))
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")

    df_clean = df.dropna(subset=required_cols).copy().reset_index(drop=True)

    if verbose:
        print(f"{'='*80}")
        print(f"FEATURE SET COMPARISON: {name_1} vs {name_2}")
        print(f"{'='*80}")

    for test_season in seasons:
        df_prior = df_clean[df_clean["season"] != test_season].copy()
        df_target = df_clean[df_clean["season"] == test_season].copy().reset_index(drop=True)

        if len(df_prior) < 10 or len(df_target) < 5:
            continue

        y_test_actuals, pred_m1_list, pred_m2_list = [], [], []
        pred_season_list, pred_opp_list, pred_fix_list = [], [], []

        for i in range(len(df_target)):
            test_match = df_target.iloc[[i]].copy()
            train_matches = pd.concat([df_prior, df_target.iloc[:i]]).copy()

            y_train = train_matches["tickets_scanned"]
            y_test_val = test_match["tickets_scanned"].values[0]

            cat_1 = train_matches[features_1].select_dtypes(exclude=[np.number]).columns.tolist()
            X_tr_1 = pd.get_dummies(train_matches[features_1], columns=cat_1, drop_first=True)
            X_te_1 = pd.get_dummies(test_match[features_1], columns=cat_1, drop_first=True).reindex(columns=X_tr_1.columns, fill_value=0)

            cat_2 = train_matches[features_2].select_dtypes(exclude=[np.number]).columns.tolist()
            X_tr_2 = pd.get_dummies(train_matches[features_2], columns=cat_2, drop_first=True)
            X_te_2 = pd.get_dummies(test_match[features_2], columns=cat_2, drop_first=True).reindex(columns=X_tr_2.columns, fill_value=0)

            rf = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
            
            rf.fit(X_tr_1, y_train)
            pred_m1_list.append(rf.predict(X_te_1)[0])
            
            rf.fit(X_tr_2, y_train)
            pred_m2_list.append(rf.predict(X_te_2)[0])

            y_test_actuals.append(y_test_val)
            pred_season_list.append(train_matches["tickets_scanned"].mean())
            pred_opp_list.append(test_match["opponent_avg"].values[0])
            pred_fix_list.append(test_match["same_fixture_baseline"].values[0])

        if not y_test_actuals: continue

        y_test = np.array(y_test_actuals)
        
        season_results.append({
            "season": test_season,
            "Baseline: Season Avg": mean_absolute_error(y_test, pred_season_list),
            "Baseline: Opponent Avg": mean_absolute_error(y_test, pred_opp_list),
            "Baseline: Same Fixture": mean_absolute_error(y_test, pred_fix_list),
            name_1: mean_absolute_error(y_test, pred_m1_list),
            name_2: mean_absolute_error(y_test, pred_m2_list)
        })
        if verbose:
            print(f"✓ Season {test_season} processed.")

    results_df = pd.DataFrame(season_results).set_index("season")
    latest_season_name = results_df.index[-1]
    
    overall_means = results_df.mean()
    results_df.loc["AVERAGE"] = overall_means

    if verbose:
        print(f"\n{'='*60}")
        print(f"{'FINAL PERFORMANCE SUMMARY':^60}")
        print(f"{'='*60}")
        print(f"{'Metric':<25} | {'Latest ('+str(latest_season_name)+')':<15} | {'Overall Avg':<12}")
        print("-" * 60)
        for col in results_df.columns:
            latest_val = results_df.loc[latest_season_name, col]
            avg_val = results_df.loc["AVERAGE", col]
            print(f"{col:<25} | {latest_val:<15.0f} | {avg_val:<12.0f}")
        print(f"{'='*60}\n")

    return results_df

# ==========================================
# 2. SEPARATED PLOTTING FUNCTION
# ==========================================
def plot_comparison_results(results_df, name_1="Model 1", name_2="Model 2"):
    """Takes the DataFrame returned by compare_feature_sets and plots it."""
    # Since we appended "AVERAGE" at the end, the latest season is the second to last index
    latest_season_name = results_df.index[-2] 
    
    cols = ["Baseline: Season Avg", "Baseline: Opponent Avg", "Baseline: Same Fixture", name_1, name_2]
    colors = ["#d9d9d9", "#bdbdbd", "#969696", "#2ca02c", "#1f77b4"]
    
    x = np.arange(len(results_df))
    width = 0.15
    fig, ax = plt.subplots(figsize=(14, 7))
    offsets = [-2*width, -width, 0, width, 2*width]

    for i, (col, color) in enumerate(zip(cols, colors)):
        if col not in results_df.columns:
            continue # Failsafe if column names change
            
        rects = ax.bar(x + offsets[i], results_df[col], width=width, label=col, color=color)
        
        for j, rect in enumerate(rects):
            row_name = results_df.index[j]
            if row_name == latest_season_name or row_name == "AVERAGE":
                height = rect.get_height()
                ax.annotate(f'{height:.0f}',
                            xy=(rect.get_x() + rect.get_width() / 2, height),
                            xytext=(0, 3),  
                            textcoords="offset points",
                            ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(results_df.index)
    
    labels = ax.get_xticklabels()
    for lbl in labels:
        if lbl.get_text() == "AVERAGE":
            lbl.set_weight("bold")
            lbl.set_color("red")
        if lbl.get_text() == str(latest_season_name):
            lbl.set_weight("bold")
            lbl.set_color("black")

    ax.set_ylabel("Mean Absolute Error (MAE)")
    ax.set_title(f"Model Performance: {name_1} vs {name_2}\n(Numbers shown for Latest Season and Global Average)")
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.grid(axis="y", alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 3. AUTOMATION LOOP TO FIND BETTER FEATURES
# ==========================================

def find_best_feature_combinations(df, base_features, candidate_features, max_features_to_add=1):
    """
    Loops through combinations of candidate features, adds them to the base model, 
    and evaluates performance on BOTH the Overall Average and the Latest Season.
    """
    print(f"Testing adding up to {max_features_to_add} features to the Base model...")
    results_list = []
    
    for k in range(1, max_features_to_add + 1):
        for combo in itertools.combinations(candidate_features, k):
            added_features = list(combo)
            test_features = base_features + added_features
            combo_name = f"+ {', '.join(added_features)}"
            
            try:
                # Run comparison SILENTLY
                res_df = compare_feature_sets(df, base_features, test_features, 
                                              name_1="Base", name_2="Test", verbose=False)
                
                # Extract the name of the latest season (it's the second to last row index)
                latest_season_name = res_df.index[-2]
                
                # Extract MAEs
                base_mae_avg = res_df.loc["AVERAGE", "Base"]
                test_mae_avg = res_df.loc["AVERAGE", "Test"]
                imp_avg = base_mae_avg - test_mae_avg
                
                base_mae_latest = res_df.loc[latest_season_name, "Base"]
                test_mae_latest = res_df.loc[latest_season_name, "Test"]
                imp_latest = base_mae_latest - test_mae_latest
                
                # Determine symbols
                sym_avg = "✅" if imp_avg > 0 else "❌"
                sym_latest = "✅" if imp_latest > 0 else "❌"
                
                # Save if it improved AT LEAST ONE of the metrics
                if imp_avg > 0 or imp_latest > 0:
                    results_list.append({
                        "Added Features": combo_name,
                        "Latest Improvement": imp_latest,
                        "Avg Improvement": imp_avg,
                        "New Latest MAE": test_mae_latest,
                        "New Avg MAE": test_mae_avg,
                        "Base Latest MAE": base_mae_latest,
                        "Base Avg MAE": base_mae_avg
                    })
                    print(f"{sym_avg} Avg | {sym_latest} Latest | {combo_name:<35} | Improved by -> Avg: {imp_avg:>6.2f} | Latest: {imp_latest:>6.2f}")
                else:
                    # Print failures so you know it's working, but keep it aligned
                    print(f"{sym_avg} Avg | {sym_latest} Latest | {combo_name:<35} | (Worse on both metrics)")
                    
            except Exception as e:
                print(f"⚠️ Error with combo {combo_name}: {e}")
                
    if not results_list:
        print("\nNo combinations performed better than the base model on either metric.")
        return pd.DataFrame()
        
    # Sort primarily by Latest Improvement, then by Average Improvement
    final_df = pd.DataFrame(results_list).sort_values(
        by=["Latest Improvement", "Avg Improvement"], ascending=[False, False]
    ).reset_index(drop=True)
    
    return final_df


def find_features_to_remove(df, current_features, max_features_to_remove=1):
    """
    Loops through combinations of current features, removes them from the model, 
    and evaluates performance on BOTH the Overall Average and the Latest Season.
    """
    print(f"Testing removing up to {max_features_to_remove} features from the model...")
    results_list = []
    
    for k in range(1, max_features_to_remove + 1):
        for combo in itertools.combinations(current_features, k):
            removed_features = list(combo)
            test_features = [f for f in current_features if f not in removed_features]
            
            # Edge case: A model needs at least 1 feature to run
            if len(test_features) == 0:
                continue
                
            combo_name = f"- {', '.join(removed_features)}"
            
            try:
                # Run comparison SILENTLY
                res_df = compare_feature_sets(df, 
                                              features_1=current_features, 
                                              features_2=test_features, 
                                              name_1="Base", 
                                              name_2="Test", 
                                              verbose=False)
                
                # Extract the name of the latest season
                latest_season_name = res_df.index[-2]
                
                # Extract MAEs
                base_mae_avg = res_df.loc["AVERAGE", "Base"]
                test_mae_avg = res_df.loc["AVERAGE", "Test"]
                imp_avg = base_mae_avg - test_mae_avg
                
                base_mae_latest = res_df.loc[latest_season_name, "Base"]
                test_mae_latest = res_df.loc[latest_season_name, "Test"]
                imp_latest = base_mae_latest - test_mae_latest
                
                # Determine symbols
                sym_avg = "✅" if imp_avg > 0 else "❌"
                sym_latest = "✅" if imp_latest > 0 else "❌"
                
                # Save if it improved AT LEAST ONE of the metrics
                if imp_avg > 0 or imp_latest > 0:
                    results_list.append({
                        "Removed Features": combo_name,
                        "Latest Improvement": imp_latest,
                        "Avg Improvement": imp_avg,
                        "New Latest MAE": test_mae_latest,
                        "New Avg MAE": test_mae_avg,
                        "Base Latest MAE": base_mae_latest,
                        "Base Avg MAE": base_mae_avg
                    })
                    print(f"{sym_avg} Avg | {sym_latest} Latest | {combo_name:<35} | Improved by -> Avg: {imp_avg:>6.2f} | Latest: {imp_latest:>6.2f}")
                else:
                    pass # Uncomment the line below if you want to see every failure
                    # print(f"{sym_avg} Avg | {sym_latest} Latest | {combo_name:<35} | (Worse on both metrics)")
                    
            except Exception as e:
                print(f"⚠️ Error with removing {combo_name}: {e}")
                
    if not results_list:
        print("\nNo feature removals performed better than the base model on either metric.")
        return pd.DataFrame()
        
    # Sort primarily by Latest Improvement, then by Average Improvement
    final_df = pd.DataFrame(results_list).sort_values(
        by=["Latest Improvement", "Avg Improvement"], ascending=[False, False]
    ).reset_index(drop=True)
    
    return final_df

#### 1. ENGINEER FEATURES

In [21]:
df = df.sort_values("match_date").reset_index(drop=True)

# --- 1. BASE NUMERICS & FIXING MISSING COLUMNS ---
df["matchday"] = pd.to_numeric(df["matchday"], errors="coerce")
df["is_weekend"] = pd.to_numeric(df["is_weekend"], errors="coerce")
df["academic_week"] = pd.to_numeric(df["academic_week"], errors="coerce")

if "tickets_trib1" in df.columns:
    df["tickets_trib1"] = pd.to_numeric(df["tickets_trib1"], errors="coerce")
else:
    df["tickets_trib1"] = np.nan

if "tickets_sold_total" in df.columns:
    df["tickets_sold_total"] = pd.to_numeric(df["tickets_sold_total"], errors="coerce")
else:
    df["tickets_sold_total"] = np.nan

# --- 2. CATEGORICALS ---
df["school_holiday_name"] = df["school_holiday_name"].fillna("None").astype(str)
df["is_public_holiday"] = df["is_public_holiday"].fillna("False").astype(str)
df["has_promotion"] = df["has_promotion"].fillna("False").astype(str)

if "stage" not in df.columns:
    df["stage"] = "None"
else:
    df["stage"] = df["stage"].fillna("None").astype(str)

# --- 3. ROLLING & EXPANDING AVERAGES ---
df["last_home_attendance"] = df["tickets_scanned"].shift(1)
df["rolling_avg_3"] = df["tickets_scanned"].shift(1).rolling(window=3).mean()
df["rolling_avg_2"] = df["tickets_scanned"].shift(1).rolling(window=2).mean()
df["rolling_avg_1"] = df["tickets_scanned"].shift(1).rolling(window=1).mean()

df["season_expanding_avg"] = df["tickets_scanned"].shift(1).expanding().mean()

# Opponent Baseline
df["opponent_avg"] = df.groupby("away_team")["tickets_scanned"].transform(lambda x: x.shift().expanding().mean())
df["opponent_avg"] = df["opponent_avg"].fillna(df["season_expanding_avg"])

# Same Fixture Baseline
df["same_fixture_baseline"] = df.groupby("away_team")["tickets_scanned"].shift(1)
df["same_fixture_baseline"] = df["same_fixture_baseline"].fillna(df["season_expanding_avg"])

# --- 4. HISTORICAL RESULTS ---
# Mapping to standard football points (3 for Win, 1 for Draw, 0 for Loss)
df["last_result_numeric"] = df["last_result_vs_opponent"].str[0].map({'W': 3, 'D': 1, 'L': 0}).fillna(0)

# Get the last 3 results against THIS specific opponent (Vectorized)
df["result_minus_1"] = df["last_result_numeric"] # The result from the immediate last matchup
df["result_minus_2"] = df.groupby("away_team")["last_result_numeric"].shift(1)
df["result_minus_3"] = df.groupby("away_team")["last_result_numeric"].shift(2)

df["points_last_3"] = (
    df["result_minus_1"].fillna(0) + 
    df["result_minus_2"].fillna(0) + 
    df["result_minus_3"].fillna(0)
)

# --- 5. FORM & MOMENTUM FEATURES ---
eps = 1e-6
df["form_vs_trend"] = df["rolling_avg_3"] - df["season_expanding_avg"]
df["form_ratio"] = df["rolling_avg_3"] / (df["season_expanding_avg"] + eps)
df["momentum_2"] = df["rolling_avg_3"] - df["last_home_attendance"]

# --- 6. TICKET/SALES RATIO FEATURES ---
df["trib1_div_trend"] = df["tickets_trib1"] / (df["season_expanding_avg"] + eps)
df["sold_div_trend"] = df["tickets_sold_total"] / (df["season_expanding_avg"] + eps)

df["trib1_div_trend"] = df["trib1_div_trend"].fillna(0)
df["sold_div_trend"] = df["sold_div_trend"].fillna(0)

# --- 7. CLEANUP ---
# Drop early season games where rolling averages can't be calculated
df = df.dropna(subset=["last_home_attendance", "rolling_avg_3"]).reset_index(drop=True)

#### 2. COMPARE MODELS WITH DIFFERENT FEATURE SETS

In [ ]:
# List of features in the base model
current_model = [
    "matchday", "is_weekend", "academic_week",
    "stage", "away_team", "rolling_avg_2", "season_expanding_avg",
    "opponent_avg"
]

# new_model = [
#     "opponent_avg", "matchday", "is_weekend", "academic_week",
#     "stage", "away_team", "rolling_avg_2", "season_expanding_avg",
# ]

# # (Optional) Run the standard single comparison & plot it manually
# results_df = compare_feature_sets(
#     df=df, 
#     features_1=current_model, 
#     features_2=new_model, 
#     name_1="Base", 
#     name_2="Iseline",
#     verbose=True
# )
# plot_comparison_results(results_df, "Base", "Iseline")


# ==========================================
# RUN THE AUTOMATED LOOP
# ==========================================
# Let's say you want to figure out which 1 or 2 features from this list 
# actually improve your current model:
candidates = [
    "rolling_avg_3", "points_last_3", 
    "form_vs_trend", "form_ratio", "momentum_2",
    "opponent_avg",
]

# Run the search
best_combos_df = find_best_feature_combinations(
    df,
    current_model,
    candidates,
    max_features_to_add=2)

# # Show the best results
# features_to_drop_df = find_features_to_remove(
#     df=df,
#     current_features=new_model,
#     max_features_to_remove=1,    # Tries removing 1 feature, then 2 features at a time
#     target_metric="AVERAGE"      # Could also use a specific season string
# )

print("\nTOP FEATURE REMOVALS FOUND:")
display(best_combos_df.head(10))

Testing adding up to 2 features to the Base model...
✅ Avg | ❌ Latest | + rolling_avg_3                     | Improved by -> Avg:   4.98 | Latest: -85.97
❌ Avg | ❌ Latest | + points_last_3                     | (Worse on both metrics)
❌ Avg | ❌ Latest | + form_vs_trend                     | (Worse on both metrics)
❌ Avg | ✅ Latest | + form_ratio                        | Improved by -> Avg: -28.68 | Latest:   6.12
✅ Avg | ❌ Latest | + momentum_2                        | Improved by -> Avg:  37.30 | Latest: -63.66
✅ Avg | ✅ Latest | + opponent_avg                      | Improved by -> Avg:   1.60 | Latest:  93.49
❌ Avg | ❌ Latest | + rolling_avg_3, points_last_3      | (Worse on both metrics)
✅ Avg | ❌ Latest | + rolling_avg_3, form_vs_trend      | Improved by -> Avg:  16.94 | Latest: -51.34
✅ Avg | ❌ Latest | + rolling_avg_3, form_ratio         | Improved by -> Avg:  17.89 | Latest: -40.57
✅ Avg | ❌ Latest | + rolling_avg_3, momentum_2         | Improved by -> Avg:  36.32 | Latest: -165

,Added Features,Latest Improvement,Avg Improvement,New Latest MAE,New Avg MAE,Base Latest MAE,Base Avg MAE
0,+ opponent_avg,93.490742,1.598082,584.757434,1199.635146,678.248176,1201.233228
1,"+ form_ratio, opponent_avg",78.873091,6.637120,599.375085,1194.596107,678.248176,1201.233228
2,"+ form_vs_trend, opponent_avg",77.902725,18.376765,600.345451,1182.856462,678.248176,1201.233228
3,"+ points_last_3, opponent_avg",72.831394,4.510591,605.416782,1196.722636,678.248176,1201.233228
4,"+ momentum_2, opponent_avg",14.002675,51.612893,664.245501,1149.620335,678.248176,1201.233228
5,+ form_ratio,6.122466,-28.679134,672.125710,1229.912361,678.248176,1201.233228
6,"+ rolling_avg_3, opponent_avg",-31.818407,9.300144,710.066582,1191.933084,678.248176,1201.233228
7,"+ rolling_avg_3, form_ratio",-40.574726,17.887040,718.822902,1183.346188,678.248176,1201.233228
8,"+ rolling_avg_3, form_vs_trend",-51.341618,16.938901,729.589794,1184.294326,678.248176,1201.233228
9,+ momentum_2,-63.662831,37.303514,741.911007,1163.929714,678.248176,1201.233228
